# مختبر اليوم الرابع — مقارنة نماذج المغادرة وتحليل الأخطاء

> هذا الدفتر مبني مباشرة من مواصفات مختبرات منافذ المعتمدة للدورة.


In [ ]:
from pathlib import Path

# يعمل محليًا داخل المستودع أو عند وضع الحزمة في مجلد مستقل
DATA_CANDIDATES = [Path("manafeth_data_package"), Path("data/raw"), Path("../../data/raw")]
DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), DATA_CANDIDATES[0])
CUSTOMERS_PATH = DATA_DIR / "manafeth_customers.parquet"
ORDERS_PATH = DATA_DIR / "manafeth_orders.parquet"
VEHICLES_PATH = DATA_DIR / "markabat_listings_sample.csv"
SHIFTED_PATH = DATA_DIR / "shifted_month.parquet"
print("DATA_DIR:", DATA_DIR.resolve())


## هدف المختبر

يقارن الطالب نموذجًا مبدئيًا ونماذج شجرية على مشكلة مغادرة العملاء الحقيقية. يتعلم اختيار المقياس الذي يناسب فئة إيجابية أقل ظهورًا، ويحلل صفوف الخطأ بدل الاحتفال برقم واحد.

## السيناريو والبيانات

تبلغ نسبة `churned_30d = 1` في حزمة منافذ نحو 14% تقريبًا؛ لذلك لا تكفي الدقة وحدها. يستخدم المختبر **متوسط الدقة (Average Precision)**، وهو مقياس يلخص منحنى الدقة–الاستدعاء، لمقارنة القدرة على ترتيب العملاء الذين قد يغادرون. يستخدم أيضًا الاستدعاء بين أعلى 20% من العملاء ترتيبًا لأن هذا يناسب سيناريو قائمة متابعة محدودة لفريق خدمة العملاء.

| النموذج | دوره في المختبر | إعداد مبدئي |
|---|---|---|
| الانحدار اللوجستي | خط أساس قابل للتفسير | `max_iter=1000` |
| شجرة القرار | نموذج أسئلة متتابعة بسيط | `max_depth=4` |
| الغابة العشوائية | تجميع أشجار للتصويت | `n_estimators=150` |
| XGBoost | تعزيز متدرج للمقارنة | `n_estimators=100` و`max_depth=3` |

## خطوات التنفيذ بالتسلسل

1. استخدم فقط `X_train` و`y_train` و`preprocessor` من اليومين السابقين في مرحلة المقارنة.
2. ضع كل نموذج داخل `Pipeline` نفسه لضمان المعالجة نفسها.
3. احسب متوسط الدقة بخمس طيات تحقق متقاطع.
4. سجل النتيجة في جدول `results_df` ولا تستخدم بيانات الاختبار لاختيار النموذج.
5. اختر النموذج المرشح بسبب مكتوب: المقياس أولًا، ثم قابلية الفهم أو استقرار النتيجة إن كانت الفروق صغيرة.
6. درب النموذج المختار على كامل التدريب، ثم احسب احتمالات المغادرة على `X_test`.
7. اعرض منحنى الدقة–الاستدعاء، ثم احسب الاستدعاء عندما نتواصل مع أعلى 20% فقط من العملاء.
8. كوّن مصفوفة التباس عند عتبة 0.50 للتدريب على قراءة الأخطاء، ثم استخرج خمسة عملاء أخطأ النموذج فيهم.

## ما يكتبه أو يشغله الطالب


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    average_precision_score, precision_recall_curve,
    ConfusionMatrixDisplay, recall_score
)
from xgboost import XGBClassifier

models = {
    "انحدار لوجستي": LogisticRegression(max_iter=1000),
    "شجرة قرار": DecisionTreeClassifier(max_depth=4, random_state=42),
    "غابة عشوائية": RandomForestClassifier(n_estimators=150, random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=100, max_depth=3,
        eval_metric="logloss", random_state=42
    )
}

comparison = []
for name, model in models.items():
    workflow = Pipeline([("prepare", preprocessor), ("model", model)])
    scores = cross_val_score(
        workflow, X_train, y_train,
        cv=5, scoring="average_precision"
    )
    comparison.append([name, scores.mean(), scores.std()])

results_df = pd.DataFrame(
    comparison,
    columns=["النموذج", "متوسط الدقة", "تغير النتيجة بين الطيات"]
).sort_values("متوسط الدقة", ascending=False)
results_df


In [ ]:
chosen_name = results_df.iloc[0]["النموذج"]
final_workflow = Pipeline([
    ("prepare", preprocessor),
    ("model", models[chosen_name])
])
final_workflow.fit(X_train, y_train)
probabilities = final_workflow.predict_proba(X_test)[:, 1]

ap = average_precision_score(y_test, probabilities)
precision, recall, _ = precision_recall_curve(y_test, probabilities)
plt.plot(recall, precision)
plt.xlabel("الاستدعاء")
plt.ylabel("الإحكام")
plt.title("منحنى الدقة–الاستدعاء")
plt.show()

contact_count = int(len(y_test) * 0.20)
top_customers = pd.DataFrame({
    "الحقيقة": y_test.to_numpy(),
    "احتمال_المغادرة": probabilities
}).sort_values("احتمال_المغادرة", ascending=False).head(contact_count)
recall_at_20 = top_customers["الحقيقة"].sum() / y_test.sum()
print("متوسط الدقة على الاختبار:", round(ap, 3))
print("الاستدعاء بين أعلى 20%:", round(recall_at_20, 3))


## النتيجة المتوقعة

ينتج جدول مقارنة لأربعة نماذج، ومنحنى دقة–استدعاء، وقيمة متوسط دقة، واستدعاء في قائمة أعلى 20%. لا تضع في السلايدات رقمًا ثابتًا للأداء؛ قد تختلف النتائج بحسب الإصدار والإعدادات. النتيجة الصحيحة هي أن يبرر الطالب الاختيار ببيانات التدريب وأن يشرح أين تظهر حالات التفويت في مصفوفة الالتباس.

## المهارات التي يراجعها الطالب

يختار الطالب مقياسًا يناسب عدم توازن الفئات، ويستخدم التحقق المتقاطع، ويقارن خطوط معالجة متكافئة، ويقرأ منحنى الدقة–الاستدعاء ومصفوفة الالتباس، ويحلل أخطاء فعلية، ويستخدم شجرة القرار والغابة العشوائية وXGBoost بصورة مدروسة.

## شرط التسليم قبل مغادرة اليوم

يسلم الطالب جدولًا بأسماء النماذج ومتوسط دقتها، ومخططًا واحدًا، وجملة تتضمن سبب اختيار النموذج، وجملة عن خطأ متكرر أو حالة تحتاج متابعة.

---
